In [1]:
import pandas as pd
import optuna
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

e:\Bootcamp\DT-CG84\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Cargamos los datos desde los archivos CSV (Loading data from CSV files)
customer_profiles = pd.read_csv('csv/customer_profiles.csv')
product_affinity = pd.read_csv('csv/product_affinity.csv')
product_interactions = pd.read_csv('csv/product_interactions.csv')


In [3]:
# Unimos los datos de clientes con sus afinidades a productos
# (Merging customer data with their product affinities)
merged_data = pd.merge(
    customer_profiles, 
    product_affinity, 
    on="customer_id"
)

In [4]:
# Convertimos las categorías de texto a números para que el modelo las entienda
# (Converting text categories to numbers so the model can understand them)
encoder = LabelEncoder()
for col in ['gender', 'location', 'preferred_category', 'product_id']:
    if col in merged_data.columns:
        merged_data[col] = encoder.fit_transform(merged_data[col])

# Seleccionamos las características que usaremos para predecir
# (Selecting features we'll use for prediction)
features = [
    "age", "gender", "income", "location", "purchase_frequency", 
    "avg_order_value", "preferred_category", "clv",
    "ingredients", "quality", "brand_loyalty", "discount_sensitivity",
    "product_id"
]


In [5]:
X = merged_data[features]
y = merged_data["affinity_score"]  # Usamos el score de afinidad directamente como target

# Dividimos los datos en conjunto de entrenamiento y prueba
# (Splitting data into training and test sets)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [6]:
def objetivo_optuna(trial):
    """
    Esta función prueba diferentes combinaciones de hiperparámetros para encontrar la mejor
    (This function tests different hyperparameter combinations to find the best one)
    """
    param = {
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 7),
        'objective': 'reg:logistic'  # Para asegurar predicciones entre 0 y 1
    }
    
    # Creamos el modelo con los parámetros sugeridos
    # (Creating model with suggested parameters)
    model = XGBRegressor(**param, random_state=42)
    
    # Evaluamos el modelo usando validación cruzada con R²
    # (Evaluating model using cross-validation with R²)
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2')
    
    return scores.mean()

# Creamos el estudio de Optuna para encontrar los mejores hiperparámetros
# (Creating Optuna study to find best hyperparameters)
study = optuna.create_study(direction="maximize")
study.optimize(objetivo_optuna, n_trials=50)

print("\nMejores hiperparámetros encontrados (Best hyperparameters found):")
print(study.best_params)

[I 2025-02-12 09:55:43,351] A new study created in memory with name: no-name-04406f72-c095-4e9d-b6f1-30ac9afaf100
[I 2025-02-12 09:55:49,896] Trial 0 finished with value: -0.002551548212663812 and parameters: {'max_depth': 5, 'learning_rate': 0.22076001227657233, 'n_estimators': 77, 'subsample': 0.7420357268584141, 'colsample_bytree': 0.6582736359823127, 'min_child_weight': 2}. Best is trial 0 with value: -0.002551548212663812.
[I 2025-02-12 09:55:56,067] Trial 1 finished with value: -0.003946241948036144 and parameters: {'max_depth': 6, 'learning_rate': 0.052495427258184735, 'n_estimators': 57, 'subsample': 0.8927164935408007, 'colsample_bytree': 0.6008184369907179, 'min_child_weight': 3}. Best is trial 0 with value: -0.002551548212663812.
[I 2025-02-12 09:56:03,868] Trial 2 finished with value: -0.0038586345563510704 and parameters: {'max_depth': 8, 'learning_rate': 0.12121400713799543, 'n_estimators': 75, 'subsample': 0.6612292187943636, 'colsample_bytree': 0.7928643082377816, 'min_


Mejores hiperparámetros encontrados (Best hyperparameters found):
{'max_depth': 3, 'learning_rate': 0.08189684650700986, 'n_estimators': 79, 'subsample': 0.807324810046135, 'colsample_bytree': 0.7823888874495815, 'min_child_weight': 5}


In [7]:
# Entrenamos el modelo final con los mejores hiperparámetros
# (Training final model with best hyperparameters)
best_params = study.best_params
best_params['objective'] = 'reg:logistic'  # Aseguramos que se mantenga la función objetivo
modelo_final = XGBRegressor(**best_params, random_state=42)
modelo_final.fit(X_train, y_train)


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.7823888874495815, device=None,
             early_stopping_rounds=None, enable_categorical=False,
             eval_metric=None, feature_types=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.08189684650700986, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=3, max_leaves=None,
             min_child_weight=5, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=79, n_jobs=None,
             num_parallel_tree=None, objective='reg:logistic', ...)

In [8]:
predicciones = modelo_final.predict(X_test)
mse = mean_squared_error(y_test, predicciones)
r2 = r2_score(y_test, predicciones)

print("\nMétricas de Evaluación (Evaluation Metrics):")
print(f"Error Cuadrático Medio (MSE): {mse:.4f}")
print(f"R² Score: {r2:.4f}")


Métricas de Evaluación (Evaluation Metrics):
Error Cuadrático Medio (MSE): 0.0036
R² Score: -0.0001


In [9]:
# Importancia de características (Feature importance)
feature_importance = pd.DataFrame({
    'feature': features,
    'importance': modelo_final.feature_importances_
})
print("\nImportancia de Características (Feature Importance):")
print(feature_importance.sort_values('importance', ascending=False))

def predecir_probabilidad_compra(datos_cliente):
    """
    Predice la probabilidad de que un cliente compre (valor entre 0 y 1)
    (Predicts the probability of a customer making a purchase - value between 0 and 1)
    
    Args:
        datos_cliente (DataFrame): DataFrame con las características del cliente
        
    Returns:
        float: Probabilidad de compra entre 0 y 1
    """
    if not all(feature in datos_cliente.columns for feature in features):
        raise ValueError(f"El DataFrame debe contener todas las características: {features}")
    
    return modelo_final.predict(datos_cliente[features])

# Ejemplo de uso (Usage example):
# nuevo_cliente = pd.DataFrame([{...}])  # Crear DataFrame con las características necesarias
# probabilidad = predecir_probabilidad_compra(nuevo_cliente)


Importancia de Características (Feature Importance):
                 feature  importance
1                 gender    0.104913
6     preferred_category    0.084411
9                quality    0.082959
8            ingredients    0.079933
2                 income    0.078967
4     purchase_frequency    0.078246
11  discount_sensitivity    0.077489
0                    age    0.072593
12            product_id    0.070556
3               location    0.070110
10         brand_loyalty    0.067397
5        avg_order_value    0.067231
7                    clv    0.065195
